# 00 · Data Preparation

Load MNIST, apply the standard normalization to `[-1, 1]`, and visualize a batch of digits. The transform here is the same one used to train the cGAN and the diffusion model, so the generators learn to produce images in the same range that a `Tanh` activation outputs.

In [ ]:
import sys
from pathlib import Path

# Allow `import data`, `import model`, etc. when running the notebook
# from the project root.
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import matplotlib.pyplot as plt

from data.dataloader import get_mnist_loaders
from utils.visualize import plot_image_grid

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## Load MNIST

In [ ]:
train_loader, test_loader = get_mnist_loaders(batch_size=64, num_workers=0)
print(f'train batches: {len(train_loader)} | test batches: {len(test_loader)}')

images, labels = next(iter(train_loader))
print('batch shape:', images.shape, '| dtype:', images.dtype)
print('value range:', images.min().item(), '..', images.max().item())
print('labels:', labels[:16].tolist())

## Visualize a batch

The images are in `[-1, 1]`; `plot_image_grid` shifts them back to `[0, 1]` for display.

In [ ]:
fig = plot_image_grid(images[:32], labels=labels[:32].tolist(), n_cols=8, title='MNIST sample batch')
plt.show()

## Per-class samples

Confirm class balance and that the conditioning labels match the images. If the dataset is imbalanced (it isn't for MNIST), the generators may still memorize the majority classes well — something to keep in mind for non-MNIST extensions.

In [ ]:
import collections

samples_per_class = collections.defaultdict(list)
for img, lab in zip(images, labels):
    if len(samples_per_class[int(lab)]) < 1:
        samples_per_class[int(lab)].append(img)
    if len(samples_per_class) == 10 and all(len(v) >= 1 for v in samples_per_class.values()):
        break

# Fill in any missing classes from the next few batches.
if not all(c in samples_per_class for c in range(10)):
    for img, lab in zip(*next(iter(train_loader))):
        samples_per_class.setdefault(int(lab), []).append(img)

grid = torch.stack([samples_per_class[c][0] for c in range(10)])
fig = plot_image_grid(grid, labels=list(range(10)), n_cols=10, title='One sample per class')
plt.show()

**Next:**
* `01_cGAN_training.ipynb` — train the Conditional GAN
* `02_diffusion_training.ipynb` — train the Conditional Diffusion model
* `03_evaluation.ipynb` — FID + downstream classifier comparison